In [1]:
# Load model directly
from transformers import AutoProcessor, AutoModelForCTC

/Users/lauracao/Documents/Code/llearning/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/lauracao/Documents/Code/llearning/.venv/lib/python3.11/site-packages/torch/compiler/__init__.py:148: FutureWarning: torch._dynamo.allow_in_graph is deprecated and will be removed in a future version. Use torch._dynamo.nonstrict_trace instead.
  return torch._dynamo.allow_in_graph(fn)


In [2]:
from transformers import VitsModel, AutoTokenizer
import torch

model = VitsModel.from_pretrained("facebook/mms-tts-rhg")
tokenizer = AutoTokenizer.from_pretrained("facebook/mms-tts-rhg")

text = ":Manúic beggún azad hísafe, ar izzot arde hók ókkol ót, fúainna hísafe foida óiye. Fottí insán óttu honó forók sára elan ot aséde tamám hók ókkol arde azadi ókkol loi fáaida goróon ór hók asé. Ar, taráre dil arde demak diyé. Ótolla, taráttu ekzon loi arekzon bái hísafe maamela goróon saá. "
inputs = tokenizer(text, return_tensors="pt")

with torch.no_grad():
    output = model(**inputs).waveform

print('done')


Loading weights: 100%|██████████| 762/762 [00:00<00:00, 9623.00it/s]


done


In [3]:
import scipy
import numpy as np

audio = output.squeeze().detach().cpu().numpy().astype(np.float32)
scipy.io.wavfile.write("techno.wav", rate=model.config.sampling_rate, data=audio)

In [5]:
from openai import OpenAI

# OpenRouter client — uses the OpenAI-compatible endpoint
openrouter_client = OpenAI(
    api_key=OPENROUTER_API,
    base_url="https://openrouter.ai/api/v1",
)

def translate_to_rohingya(text: str) -> str:
    """Translate text to Rohingya Latin script using Gemini 2.5 via OpenRouter."""
    response = openrouter_client.chat.completions.create(
        model="openrouter/healer-alpha",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a translation assistant. "
                    "Translate the user's text into Rohingya written in Latin script (Rohingyalish). "
                    "Return only the translated text with no explanations or extra formatting."
                ),
            },
            {"role": "user", "content": text},
        ],
    )
    return response.choices[0].message.content.strip()

english_text = "All human beings are born free and equal in dignity and rights."
rohingya_text = translate_to_rohingya(english_text)
print(f"English : {english_text}")
print(f"Rohingya: {rohingya_text}")

English : All human beings are born free and equal in dignity and rights.
Rohingya: Sakal manusher janma hoi azad ar dignity ebong odhikar-e saman.


In [6]:
# Run TTS on the translated Rohingya text
inputs_translated = tokenizer(rohingya_text, return_tensors="pt")

with torch.no_grad():
    output_translated = model(**inputs_translated).waveform

audio_translated = output_translated.squeeze().detach().cpu().numpy().astype(np.float32)
scipy.io.wavfile.write("translated_rohingya.wav", rate=model.config.sampling_rate, data=audio_translated)
print("Saved translated_rohingya.wav")

Saved translated_rohingya.wav


In [8]:
from transformers import Wav2Vec2ForCTC, Wav2Vec2Processor
import torch
import numpy as np
import functools

ASR_MODEL_ID = "facebook/mms-1b-all"
ROHINGYA_LANG = "rhg"

@functools.lru_cache(maxsize=1)
def load_rohingya_asr_model():
    """Load MMS-1b-all ASR model with Rohingya adapter (cached)."""
    asr_processor = Wav2Vec2Processor.from_pretrained(ASR_MODEL_ID)
    asr_model = Wav2Vec2ForCTC.from_pretrained(ASR_MODEL_ID)
    asr_processor.tokenizer.set_target_lang(ROHINGYA_LANG)
    asr_model.load_adapter(ROHINGYA_LANG)
    asr_model.eval()
    return asr_processor, asr_model


def transcribe_rohingya(audio: np.ndarray, sample_rate: int) -> str:
    """Convert Rohingya speech (numpy float32 array) to Rohingya Latin text."""
    asr_processor, asr_model = load_rohingya_asr_model()
    inputs = asr_processor(audio, sampling_rate=sample_rate, return_tensors="pt")
    with torch.no_grad():
        logits = asr_model(**inputs).logits
    predicted_ids = torch.argmax(logits, dim=-1)
    return asr_processor.decode(predicted_ids[0])


def translate_audio_to_english(audio: np.ndarray, sample_rate: int) -> str:
    """
    Takes a Rohingya speech audio array and returns the English translation.

    Steps:
      1. ASR  : audio → Rohingya Latin text  (facebook/mms-1b-all + rhg adapter)
      2. MT   : Rohingya text → English      (OpenRouter)
    """
    rohingya_text = transcribe_rohingya(audio, sample_rate)
    print(f"Transcribed Rohingya: {rohingya_text}")

    response = openrouter_client.chat.completions.create(
        model="openrouter/healer-alpha",
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a translation assistant. "
                    "Translate the following Rohingya Latin script (Rohingyalish) text into English. "
                    "Return only the translated text with no explanations or extra formatting."
                ),
            },
            {"role": "user", "content": rohingya_text},
        ],
    )
    return response.choices[0].message.content.strip()


# --- Test with the audio we already generated ---
english_result = translate_audio_to_english(audio_translated, model.config.sampling_rate)
print(f"English translation: {english_result}")

Loading weights: 100%|██████████| 1096/1096 [00:00<00:00, 1430.97it/s]


Transcribed Rohingya: satal manúcer jánna hoói azat ár din níti hébáu modikarére saman
English translation: Humans are always free, yet bound by their daily routines.
